This script implements an automated extraction and preprocessing pipeline for multilingual policy documents sourced from PDFs across four countries: Ireland, France, USA, and Australia. It utilizes `pdfplumber` with layout-aware extraction to ensure text integrity, followed by a custom sentence-based chunking algorithm. This algorithm divides documents into 600-word segments with a 100-word overlap, specifically designed to preserve semantic coherence and context across chunk boundaries for advanced NLP analysis. Finally, it aggregates the processed chunks with metadata (Country, Title, Chunk ID) into a structured CSV file, preparing a clean, high-quality corpus suitable for downstream tasks such as topic modeling and semantic clustering with models like BERT and LDA.

In [1]:
import os
import pandas as pd
import pdfplumber
import nltk
from nltk.tokenize import sent_tokenize

NLTK_PATH = "../../../data/auto/"
nltk.data.path.append(NLTK_PATH)

# Configure NLTK for local environment
try:
    nltk.download('punkt', download_dir=NLTK_PATH, quiet=True)
    nltk.download('punkt_tab', download_dir=NLTK_PATH, quiet=True)
except Exception as e:
    print(f"NLTK Download warning: {e}")

In [2]:
def chunk_text_by_sentences(text, chunk_size_words=600, overlap_words=100):
    """
    Splits text into chunks based on sentence boundaries and word count.
    Includes an overlap between chunks to preserve context.
    """
    if not text or len(text.strip()) < 10:
        return []

    try:
        sentences = sent_tokenize(text)
    except LookupError:
        sentences = text.split('. ')

    chunks = []
    current_chunk = []
    current_word_count = 0

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence: 
            continue
            
        words_in_sentence = len(sentence.split())

        if current_word_count + words_in_sentence > chunk_size_words and current_chunk:
            chunks.append(" ".join(current_chunk))

            # OVERLAP LOGIC: Preserves context between chunks
            overlap_chunk = []
            overlap_count = 0
            
            for sent in reversed(current_chunk):
                sent_len = len(sent.split())
                if overlap_count + sent_len <= overlap_words:
                    overlap_chunk.insert(0, sent)
                    overlap_count += sent_len
                else:
                    break
            
            current_chunk = overlap_chunk
            current_word_count = overlap_count

        current_chunk.append(sentence)
        current_word_count += words_in_sentence

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

In [3]:
def extract_text_from_pdfs(folder_path, country_name, data_list, file_counter):
    for dirpath, dirnames, filenames in os.walk(folder_path):
        for filename in filenames:
            if filename.lower().endswith('.pdf'):
                full_pdf_path = os.path.join(dirpath, filename)
                
                print(f"[{file_counter[0]}] Processing [{country_name}]: {filename}...")
                file_counter[0] += 1
                
                try:
                    with pdfplumber.open(full_pdf_path) as pdf:
                        full_text = []
                        
                        for page in pdf.pages:
                            # SUITABLE FOR BERT+SLM & LDA: Uses layout=True to preserve reading order (e.g. multi-column text)
                            # This ensures semantic coherence which is required for high-quality embeddings and topic models.
                            text = page.extract_text(layout=True)
                            
                            if text:
                                text = text.replace('\n', ' ')
                                text = ' '.join(text.split()) 
                                full_text.append(text)
                        
                        document_content = " ".join(full_text)
                        
                        if len(document_content) < 100:
                            continue

                        text_chunks = chunk_text_by_sentences(
                            document_content, 
                            chunk_size_words=600,
                            overlap_words=100
                        )
                        
                        for i, chunk in enumerate(text_chunks):
                            if len(chunk) > 50:
                                data_list.append({
                                    'Source_File': filename,
                                    'Title': filename,
                                    'Content': chunk,     # REQUIRED FOR LDA: Input text
                                    'Country': country_name,
                                    'Chunk_ID': i + 1
                                })
                                
                except Exception as e:
                    print(f"Error reading {filename}: {e}")

In [5]:
base_folder = '../../../pdfs/policy/'

countries = {
    'Ireland': 'ireland',
    'France': 'france',
    'USA': 'usa',
    'Australia': 'australia'
}

all_docs_chunked = []
output_csv_name = '../../../data/policy/combined_policy_docs_chunked.csv'

file_counter = [1]

for country, folder_name in countries.items():
    folder_path = os.path.join(base_folder, folder_name)
    
    if os.path.exists(folder_path):
        extract_text_from_pdfs(folder_path, country, all_docs_chunked, file_counter)
    else:
        print(f"Warning: The folder '{folder_path}' does not exist. Skipping {country}.")

if all_docs_chunked:
    df = pd.DataFrame(all_docs_chunked)
    
    output_dir = os.path.dirname(output_csv_name)
    if output_dir and not os.path.exists(output_dir):
        os.makedirs(output_dir)

    try:
        df.to_csv(output_csv_name, index=False, encoding='utf-8-sig')
        print(f"\nSuccess! Chunked CSV saved to: {output_csv_name}")
        print(f"Total rows (chunks) created: {len(df)}")
        print(df['Country'].value_counts())
    except Exception as e:
        print(f"\nError saving CSV: {e}")
else:
    print("No data extracted.")

[1] Processing [Ireland]: digital-strategy-for-schools-to-2027.pdf...
[2] Processing [Ireland]: OECD_Education_Policy_Perspectives_125.pdf...
[3] Processing [Ireland]: 2026-01-20_opening-statement-sile-maguire-director-general-of-corporate-services-division-et-al-department-of-foreign-affairs-and-trade_en.pdf...
[4] Processing [Ireland]: 2025-12-16_first-interim-report_en.pdf...
[5] Processing [Ireland]: Ratified-AI-in-Assessment-Guidelines-for-Learners-v1.1.pdf...
[6] Processing [Ireland]: AI+Policy+June+2025.pdf...
[7] Processing [Ireland]: OECD_Education_Working_Papers_No._336.pdf...
[8] Processing [Ireland]: 2026-01-20_opening-statement-graham-long-chief-executive-et-al-citizens-information-board_en.pdf...
[9] Processing [Ireland]: Using_Generative_AI.pdf...
[10] Processing [Ireland]: 2026-01-20_opening-statement-ruth-kennedy-revenue-commissioner-et-al-office-of-the-revenue-commissioners_en.pdf...
[11] Processing [Ireland]: 2025-11-18_opening-statement-dr-deepak-padmanabhan-senior-

Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


[31] Processing [USA]: Developing Policy and Protocols for the use of Generative AI in K-12 Classrooms_March 2025.pdf...
[32] Processing [USA]: AI-Policy-April-2025-v2.pdf...
[33] Processing [USA]: AI-Policy-Considerations-for-schools-and-MATs.pdf...
[34] Processing [USA]: AI+Adoption+Roadmap+(1080+×+1920+px)+(2).pdf...
[35] Processing [USA]: ai_report_us_gov.pdf...
[36] Processing [USA]: ai_policy_landscape_united_states.pdf...
[37] Processing [USA]: Final Draft (3).pdf...
[38] Processing [USA]: Four States Guiding Principles for AI in Education (2).pdf...
[39] Processing [USA]: ai-guidance.pdf...
[40] Processing [USA]: Generative Artificial Intelligence (AI) in K-12 Classrooms v2.pdf...
[41] Processing [USA]: AI-in-Education-1-pager.pdf...
[42] Processing [USA]: GuideToAIInSchools.pdf...
[43] Processing [USA]: NENUFSD K-12 Artificial Intelligence (AI) Use Guidelines.pdf...
[44] Processing [USA]: AI+GUIDE+FOR+STUDENTS+(Flyer+(Portrait+8.5+×+11+in))+(1).pdf_2.pdf...
[45] Processing [US